# LFC Embedding Performance Tables

This notebook evaluates restricted LFC benchmark results for SciPlex and Tahoe. It reports descriptive L2 statistics in two ways:

- Across matched cell-line/fold units.
- Across folds only, after averaging cell lines within each fold.

These are split/cell-line uncertainty estimates, not seed-level errors.


In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

In [2]:
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "results" / "scores").exists() and (p / "results" / "metadata" / "fig_index.json").exists()
)

score_paths = {
    "SciPlex LFC": repo_root / "results" / "scores" / "sciplex_lfc_restricted.csv",
    "Tahoe LFC": repo_root / "results" / "scores" / "tahoe_lfc_restricted.csv",
}

with open(repo_root / "results" / "metadata" / "fig_index.json", "r") as f:
    fig_index = json.load(f)

for label, path in score_paths.items():
    print(label, path, "exists=", path.exists())

SciPlex LFC /Users/matthewmella/repos/foundation-models-perturbation/results/scores/sciplex_lfc_restricted.csv exists= True
Tahoe LFC /Users/matthewmella/repos/foundation-models-perturbation/results/scores/tahoe_lfc_restricted.csv exists= True


## Helpers

In [3]:
def parse_model(name):
    name = str(name)
    if name.startswith("knn_baseline_"):
        return "knn", name[len("knn_baseline_"):]
    if name.startswith("lasso_baseline_"):
        return "lasso", name[len("lasso_baseline_"):]
    return None, name


def method_map(section, estimator):
    mapping = {
        key.replace("HEAD_TYPE", estimator): value
        for key, value in fig_index[section].items()
    }
    if section == "tahoe_lfc":
        mapping.update({
            f"{estimator}_baseline_LPM_emb": ["LPM", "Response Embedding"],
            f"{estimator}_baseline_ECFP:2_pkl": ["ECFP:2 (pkl)", "Molecule Structure"],
        })
    elif section == "sciplex_lfc":
        mapping.update({
            f"{estimator}_baseline_LPM_emb": ["LPM", "Response Embedding"],
            f"{estimator}_baseline_ECFP:2_pkl": ["ECFP:2", "Molecule Structure"],
            f"{estimator}_baseline_cats2D": ["CATS2D", "Molecule Structure"],
            f"{estimator}_baseline_cats3D": ["CATS3D", "Molecule Structure"],
            f"{estimator}_baseline_fcfp": ["FCFP", "Molecule Structure"],
        })
    return mapping


def prepare_model_frame(df, section, estimator):
    mapping = method_map(section, estimator)
    out = df.copy()
    out[["estimator", "embedding_key"]] = out["name"].apply(lambda x: pd.Series(parse_model(x)))
    out = out[out["estimator"].eq(estimator)].copy()
    out = out[out["name"].isin(mapping)].copy()
    out["embedding"] = out["name"].map(lambda x: mapping[x][0])
    out["model_type"] = out["name"].map(lambda x: mapping[x][1])
    out["unit"] = out["fold"].astype(str)
    return out


def summarize_values(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    n = len(values)
    sd = values.std(ddof=1) if n > 1 else np.nan
    sem = sd / math.sqrt(n) if n > 1 else np.nan
    ci_half_width = stats.t.ppf(0.975, n - 1) * sem if n > 1 else np.nan
    return {
        "mean_l2": values.mean() if n else np.nan,
        "median_l2": np.median(values) if n else np.nan,
        "sd": sd,
        "sem": sem,
        "ci95_half_width": ci_half_width,
    }


def summarize_embeddings(df, section, estimator):
    model_df = prepare_model_frame(df, section, estimator)
    rows = []
    for name, group in model_df.groupby("name"):
        values = group["L2"].dropna().to_numpy(float)
        rows.append({
            "name": name,
            "embedding": group["embedding"].iloc[0],
            "model_type": group["model_type"].iloc[0],
            "n_units": len(values),
            **summarize_values(values),
        })
    summary = pd.DataFrame(rows).sort_values("mean_l2").reset_index(drop=True)
    summary.insert(0, "rank", np.arange(1, len(summary) + 1))
    return summary


def summarize_embeddings_by_fold(df, section, estimator):
    model_df = prepare_model_frame(df, section, estimator)
    model_df["fold_id"] = model_df["fold"].astype(str).str.rsplit(".", n=1).str[-1]
    fold_means = (
        model_df
        .groupby(["name", "embedding", "model_type", "fold_id"], as_index=False)["L2"]
        .mean()
    )
    rows = []
    for name, group in fold_means.groupby("name"):
        values = group["L2"].dropna().to_numpy(float)
        rows.append({
            "name": name,
            "embedding": group["embedding"].iloc[0],
            "model_type": group["model_type"].iloc[0],
            "n_folds": len(values),
            **summarize_values(values),
        })
    summary = pd.DataFrame(rows).sort_values("mean_l2").reset_index(drop=True)
    summary.insert(0, "rank", np.arange(1, len(summary) + 1))
    return summary


def compare_heads(df):
    out = df.copy()
    out[["estimator", "embedding_key"]] = out["name"].apply(lambda x: pd.Series(parse_model(x)))
    out = out[out["estimator"].isin(["knn", "lasso"])].copy()
    comparison = out.groupby(["embedding_key", "estimator"])["L2"].mean().unstack().dropna()
    comparison["best_head"] = np.where(comparison["knn"] <= comparison["lasso"], "knn", "lasso")
    comparison["lasso_minus_knn"] = comparison["lasso"] - comparison["knn"]
    return comparison.sort_values("lasso_minus_knn").reset_index()


In [4]:
def markdown_table(frame):
    text = frame.copy().astype(str)
    columns = list(text.columns)
    rows = text.to_numpy().tolist()
    widths = [len(col) for col in columns]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(value))

    def format_row(row):
        return "| " + " | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)) + " |"

    header = format_row(columns)
    separator = "| " + " | ".join("-" * width for width in widths) + " |"
    body = [format_row(row) for row in rows]
    return "\n".join([header, separator, *body])

## Compute Tables

In [5]:
datasets = {
    "SciPlex LFC": {
        "path": score_paths["SciPlex LFC"],
        "section": "sciplex_lfc",
    },
    "Tahoe LFC": {
        "path": score_paths["Tahoe LFC"],
        "section": "tahoe_lfc",
    },
}

tables = {}
fold_tables = {}
head_comparisons = {}

for dataset_name, info in datasets.items():
    scores = pd.read_csv(info["path"], index_col=0)
    for estimator in ["knn", "lasso"]:
        tables[(dataset_name, estimator)] = summarize_embeddings(scores, info["section"], estimator)
        fold_tables[(dataset_name, estimator)] = summarize_embeddings_by_fold(scores, info["section"], estimator)
    head_comparisons[dataset_name] = compare_heads(scores)

list(tables), list(fold_tables)


([('SciPlex LFC', 'knn'),
  ('SciPlex LFC', 'lasso'),
  ('Tahoe LFC', 'knn'),
  ('Tahoe LFC', 'lasso')],
 [('SciPlex LFC', 'knn'),
  ('SciPlex LFC', 'lasso'),
  ('Tahoe LFC', 'knn'),
  ('Tahoe LFC', 'lasso')])

## SciPlex LFC

In [6]:
display(Markdown("### SciPlex LFC KNN: cell-line/fold units"))
display(tables[("SciPlex LFC", "knn")])


display(Markdown("### SciPlex LFC KNN: fold-only variation"))
display(fold_tables[("SciPlex LFC", "knn")])


display(Markdown("### SciPlex LFC Lasso: cell-line/fold units"))
display(tables[("SciPlex LFC", "lasso")])


display(Markdown("### SciPlex LFC Lasso: fold-only variation"))
display(fold_tables[("SciPlex LFC", "lasso")])


display(Markdown("### SciPlex KNN vs Lasso"))
display(head_comparisons["SciPlex LFC"])


### SciPlex LFC KNN: cell-line/fold units

,rank,name,embedding,model_type,n_units,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,knn_baseline_pca,Idealized Baseline,Positive Control,15,5.804627,6.052330,0.910188,0.235009,0.504045
1,2,knn_baseline_LPM_emb,LPM,Response Embedding,15,6.852237,7.148205,0.989869,0.255583,0.548171
2,3,knn_baseline_erg,ErG,Molecule Structure,15,6.956262,7.307665,0.976051,0.252015,0.540519
3,4,knn_baseline_cats2D,CATS2D,Molecule Structure,15,6.958345,7.233543,0.918466,0.237147,0.508630
4,5,knn_baseline_ChemBERTa-77M-MTR,ChemBERTa-77M-MTR,SMILES Transformer,15,6.979861,7.203293,0.902210,0.232950,0.499628
5,6,knn_baseline_maccs,MACCS,Molecule Structure,15,6.997721,7.267664,0.817485,0.211074,0.452708
6,7,knn_baseline_chatgpt,ChatGPT,LLM,15,6.999440,6.970930,0.721586,0.186313,0.399601
7,8,knn_baseline_cats3D,CATS3D,Molecule Structure,15,7.021723,7.288952,0.892638,0.230478,0.494327
8,9,knn_baseline_random,Random Embeddings,Negative Control,15,7.072853,7.319257,0.964705,0.249086,0.534236
9,10,knn_baseline_avalon,Avalon,Molecule Structure,15,7.089507,7.339905,0.933225,0.240958,0.516803


### SciPlex LFC KNN: fold-only variation

,rank,name,embedding,model_type,n_folds,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,knn_baseline_pca,Idealized Baseline,Positive Control,5,5.804627,6.193429,0.900312,0.402632,1.117885
1,2,knn_baseline_LPM_emb,LPM,Response Embedding,5,6.852237,7.323395,0.972122,0.434746,1.207049
2,3,knn_baseline_erg,ErG,Molecule Structure,5,6.956262,7.460647,0.978651,0.437666,1.215156
3,4,knn_baseline_cats2D,CATS2D,Molecule Structure,5,6.958345,7.374700,0.921118,0.411937,1.143719
4,5,knn_baseline_ChemBERTa-77M-MTR,ChemBERTa-77M-MTR,SMILES Transformer,5,6.979861,7.327861,0.890709,0.398337,1.105962
5,6,knn_baseline_maccs,MACCS,Molecule Structure,5,6.997721,7.386377,0.799204,0.357415,0.992343
6,7,knn_baseline_chatgpt,ChatGPT,LLM,5,6.999440,7.125132,0.640604,0.286487,0.795414
7,8,knn_baseline_cats3D,CATS3D,Molecule Structure,5,7.021723,7.456204,0.885249,0.395895,1.099181
8,9,knn_baseline_random,Random Embeddings,Negative Control,5,7.072853,7.546501,0.963784,0.431017,1.196696
9,10,knn_baseline_avalon,Avalon,Molecule Structure,5,7.089507,7.516533,0.924887,0.413622,1.148399


### SciPlex LFC Lasso: cell-line/fold units

,rank,name,embedding,model_type,n_units,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,lasso_baseline_pca,Idealized Baseline,Positive Control,15,4.669797,5.009134,0.795060,0.205284,0.440289
1,2,lasso_baseline_LPM_emb,LPM,Response Embedding,15,6.784516,7.044391,0.915627,0.236414,0.507057
2,3,lasso_baseline_chatgpt,ChatGPT,LLM,15,6.896277,7.133454,0.727976,0.187963,0.403140
3,4,lasso_baseline_fcfp,FCFP,Molecule Structure,15,6.989228,7.237117,0.906645,0.234095,0.502083
4,5,lasso_baseline_secfp,SECFP,Molecule Structure,15,7.023113,7.233976,0.858824,0.221747,0.475601
5,6,lasso_baseline_topological,Topological,Molecule Structure,15,7.029225,7.291337,0.948195,0.244823,0.525093
6,7,lasso_baseline_ECFP:2_pkl,ECFP:2,Molecule Structure,15,7.039215,7.267128,0.923557,0.238461,0.511449
7,8,lasso_baseline_maccs,MACCS,Molecule Structure,15,7.041672,7.247529,0.890967,0.230047,0.493401
8,9,lasso_baseline_erg,ErG,Molecule Structure,15,7.043638,7.349165,0.935620,0.241576,0.518129
9,10,lasso_baseline_random,Random Embeddings,Negative Control,15,7.058822,7.308034,0.962915,0.248624,0.533245


### SciPlex LFC Lasso: fold-only variation

,rank,name,embedding,model_type,n_folds,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,lasso_baseline_pca,Idealized Baseline,Positive Control,5,4.669797,4.715858,0.537173,0.240231,0.666988
1,2,lasso_baseline_LPM_emb,LPM,Response Embedding,5,6.784516,7.174154,0.898755,0.401935,1.115951
2,3,lasso_baseline_chatgpt,ChatGPT,LLM,5,6.896277,7.155331,0.691117,0.309077,0.858135
3,4,lasso_baseline_fcfp,FCFP,Molecule Structure,5,6.989228,7.387786,0.885512,0.396013,1.099508
4,5,lasso_baseline_secfp,SECFP,Molecule Structure,5,7.023113,7.400512,0.837915,0.374727,1.040409
5,6,lasso_baseline_topological,Topological,Molecule Structure,5,7.029225,7.509913,0.948995,0.424403,1.178333
6,7,lasso_baseline_ECFP:2_pkl,ECFP:2,Molecule Structure,5,7.039215,7.431582,0.903930,0.404250,1.122377
7,8,lasso_baseline_maccs,MACCS,Molecule Structure,5,7.041672,7.464692,0.880598,0.393816,1.093407
8,9,lasso_baseline_erg,ErG,Molecule Structure,5,7.043638,7.465145,0.933236,0.417356,1.158766
9,10,lasso_baseline_random,Random Embeddings,Negative Control,5,7.058822,7.530682,0.963442,0.430865,1.196272


### SciPlex KNN vs Lasso

estimator,embedding_key,knn,lasso,best_head,lasso_minus_knn
0,pca,5.804627,4.669797,lasso,-1.134830
1,fcfp,7.101159,6.989228,lasso,-0.111931
2,ChemBERTa-77M-MLM,7.188733,7.078321,lasso,-0.110412
3,chatgpt,6.999440,6.896277,lasso,-0.103163
4,secfp,7.107513,7.023113,lasso,-0.084400
5,topological,7.101024,7.029225,lasso,-0.071798
6,LPM_emb,6.852237,6.784516,lasso,-0.067721
7,ECFP:2_pkl,7.093760,7.039215,lasso,-0.054545
8,random,7.072853,7.058822,lasso,-0.014031
9,avalon,7.089507,7.077558,lasso,-0.011949


## Tahoe LFC

In [7]:
display(Markdown("### Tahoe LFC KNN: cell-line/fold units"))
display(tables[("Tahoe LFC", "knn")])


display(Markdown("### Tahoe LFC KNN: fold-only variation"))
display(fold_tables[("Tahoe LFC", "knn")])


display(Markdown("### Tahoe LFC Lasso: cell-line/fold units"))
display(tables[("Tahoe LFC", "lasso")])


display(Markdown("### Tahoe LFC Lasso: fold-only variation"))
display(fold_tables[("Tahoe LFC", "lasso")])


display(Markdown("### Tahoe KNN vs Lasso"))
display(head_comparisons["Tahoe LFC"])


### Tahoe LFC KNN: cell-line/fold units

,rank,name,embedding,model_type,n_units,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,knn_baseline_pca,Idealized Baseline,Positive Control,225,4.063281,4.059013,0.338179,0.022545,0.044428
1,2,knn_baseline_LPM_emb,LPM,Response Embedding,225,4.770296,4.770220,0.369516,0.024634,0.048545
2,3,knn_baseline_AIDOcell_100M_Norman_Aligned_(D=6...,AIDO.Cell 100M - Norman,Gene Target,225,5.792001,5.788429,0.438408,0.029227,0.057595
3,4,knn_baseline_boltz_affinity_probability_binary...,Boltz Binding Probability (Protein),Protein Affinity,225,5.800154,5.800365,0.439116,0.029274,0.057688
4,5,knn_baseline_MiniMol,MiniMol,SMILES Transformer,225,5.800285,5.787052,0.436783,0.029119,0.057382
5,6,knn_baseline_gene_targets_binary_name_only,Targets Binary (Name),Gene Target,225,5.810383,5.808743,0.438598,0.029240,0.057620
6,7,knn_baseline_gene_targets_confidence_with_pubchem,"Targets Weighted (Name, PubChem)",Gene Target,225,5.812515,5.799079,0.438723,0.029248,0.057637
7,8,knn_baseline_scPRINT_Norman_K-562_controls_(D=...,scPRINT - Norman,Gene Target,225,5.813289,5.818176,0.444720,0.029648,0.058425
8,9,knn_baseline_erg,ErG,Molecule Structure,225,5.814224,5.807973,0.437204,0.029147,0.057437
9,10,knn_baseline_avalon,Avalon,Molecule Structure,225,5.816644,5.815047,0.439838,0.029323,0.057783


### Tahoe LFC KNN: fold-only variation

,rank,name,embedding,model_type,n_folds,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,knn_baseline_pca,Idealized Baseline,Positive Control,5,4.063281,4.101503,0.107852,0.048233,0.133916
1,2,knn_baseline_LPM_emb,LPM,Response Embedding,5,4.770296,4.798161,0.139710,0.062480,0.173473
2,3,knn_baseline_AIDOcell_100M_Norman_Aligned_(D=6...,AIDO.Cell 100M - Norman,Gene Target,5,5.792001,5.854673,0.152553,0.068224,0.189420
3,4,knn_baseline_boltz_affinity_probability_binary...,Boltz Binding Probability (Protein),Protein Affinity,5,5.800154,5.856124,0.149980,0.067073,0.186225
4,5,knn_baseline_MiniMol,MiniMol,SMILES Transformer,5,5.800285,5.857021,0.139088,0.062202,0.172700
5,6,knn_baseline_gene_targets_binary_name_only,Targets Binary (Name),Gene Target,5,5.810383,5.857917,0.145819,0.065212,0.181058
6,7,knn_baseline_gene_targets_confidence_with_pubchem,"Targets Weighted (Name, PubChem)",Gene Target,5,5.812515,5.878846,0.147842,0.066117,0.183570
7,8,knn_baseline_scPRINT_Norman_K-562_controls_(D=...,scPRINT - Norman,Gene Target,5,5.813289,5.896659,0.166662,0.074534,0.206938
8,9,knn_baseline_erg,ErG,Molecule Structure,5,5.814224,5.878842,0.139167,0.062237,0.172798
9,10,knn_baseline_avalon,Avalon,Molecule Structure,5,5.816644,5.856822,0.155033,0.069333,0.192499


### Tahoe LFC Lasso: cell-line/fold units

,rank,name,embedding,model_type,n_units,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,lasso_baseline_pca,Idealized Baseline,Positive Control,225,2.724547,2.669524,0.383196,0.025546,0.050342
1,2,lasso_baseline_LPM_emb,LPM,Response Embedding,225,4.564811,4.550392,0.344183,0.022946,0.045217
2,3,lasso_baseline_chatgpt,ChatGPT,LLM,225,5.783712,5.781216,0.435656,0.029044,0.057234
3,4,lasso_baseline_AIDOcell_100M_Norman_Aligned_(D...,AIDO.Cell 100M - Norman,Gene Target,225,5.803678,5.803602,0.436501,0.029100,0.057345
4,5,lasso_baseline_MiniMol,MiniMol,SMILES Transformer,225,5.811721,5.804034,0.436574,0.029105,0.057354
5,6,lasso_baseline_boltz_affinity_probability_bina...,Boltz Binding Probability (Protein),Protein Affinity,225,5.812454,5.804588,0.438358,0.029224,0.057589
6,7,lasso_baseline_boltz_affinity_pred_value_protein,Boltz (Protein),Protein Affinity,225,5.816060,5.814659,0.436971,0.029131,0.057407
7,8,lasso_baseline_scPRINT_Norman_K-562_controls_(...,scPRINT - Norman,Gene Target,225,5.816909,5.815091,0.440112,0.029341,0.057819
8,9,lasso_baseline_boltz_affinity_pred_value_fragment,Boltz (Fragment),Protein Affinity,225,5.817068,5.812417,0.436600,0.029107,0.057358
9,10,lasso_baseline_MolT5,MolT5,SMILES Transformer,225,5.817237,5.820357,0.437282,0.029152,0.057447


### Tahoe LFC Lasso: fold-only variation

,rank,name,embedding,model_type,n_folds,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,lasso_baseline_pca,Idealized Baseline,Positive Control,5,2.724547,2.704874,0.086818,0.038826,0.107799
1,2,lasso_baseline_LPM_emb,LPM,Response Embedding,5,4.564811,4.579994,0.104345,0.046665,0.129562
2,3,lasso_baseline_chatgpt,ChatGPT,LLM,5,5.783712,5.840874,0.146502,0.065518,0.181906
3,4,lasso_baseline_AIDOcell_100M_Norman_Aligned_(D...,AIDO.Cell 100M - Norman,Gene Target,5,5.803678,5.849905,0.146191,0.065379,0.181520
4,5,lasso_baseline_MiniMol,MiniMol,SMILES Transformer,5,5.811721,5.857904,0.139508,0.062390,0.173222
5,6,lasso_baseline_boltz_affinity_probability_bina...,Boltz Binding Probability (Protein),Protein Affinity,5,5.812454,5.864220,0.143055,0.063976,0.177626
6,7,lasso_baseline_boltz_affinity_pred_value_protein,Boltz (Protein),Protein Affinity,5,5.816060,5.861075,0.139895,0.062563,0.173703
7,8,lasso_baseline_scPRINT_Norman_K-562_controls_(...,scPRINT - Norman,Gene Target,5,5.816909,5.872353,0.143884,0.064347,0.178656
8,9,lasso_baseline_boltz_affinity_pred_value_fragment,Boltz (Fragment),Protein Affinity,5,5.817068,5.867147,0.136898,0.061223,0.169982
9,10,lasso_baseline_MolT5,MolT5,SMILES Transformer,5,5.817237,5.873280,0.141344,0.063211,0.175502


### Tahoe KNN vs Lasso

estimator,embedding_key,knn,lasso,best_head,lasso_minus_knn
0,pca,4.063281,2.724547,lasso,-1.338734
1,LPM_emb,4.770296,4.564811,lasso,-0.205485
2,ChemBERTa-77M-MLM,5.898490,5.820648,lasso,-0.077843
3,chatgpt,5.838794,5.783712,lasso,-0.055083
4,MolT5,5.852688,5.817237,lasso,-0.035451
5,unimol2-570m-H,5.852621,5.823838,lasso,-0.028782
6,random,5.849788,5.823871,lasso,-0.025917
7,boltz_affinity_pred_value_protein,5.840819,5.816060,lasso,-0.024759
8,boltz_affinity_pred_value_fragment,5.836141,5.817068,lasso,-0.019073
9,ChemBERTa-77M-MTR,5.841255,5.823302,lasso,-0.017953


## Export Markdown Report

In [8]:
def format_report_table(table, count_col="n_units"):
    cols = [
        "rank", "embedding", "model_type", count_col, "mean_l2", "median_l2",
        "sd", "sem", "ci95_half_width",
    ]
    out = table[cols].copy()
    numeric_cols = ["mean_l2", "median_l2", "sd", "sem", "ci95_half_width"]
    for col in numeric_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
    return markdown_table(out)


report_parts = [
    "# LFC Embedding Performance Report",
    "Lower `L2` is better. The main tables compute `SD`, `SEM`, and `95% CI +/-` across matched cell-line/fold units, not seed-level repeats.",
    "The fold-only tables first average cell lines within each fold for each embedding, then compute the same descriptive statistics across folds.",
]

for dataset_name in ["SciPlex LFC", "Tahoe LFC"]:
    report_parts.append(f"## {dataset_name}")
    for estimator in ["knn", "lasso"]:
        table = tables[(dataset_name, estimator)]
        fold_table = fold_tables[(dataset_name, estimator)]
        report_parts.append(f"### {dataset_name} {estimator.upper()} - cell-line/fold variation")
        report_parts.append(format_report_table(table, "n_units"))
        report_parts.append(f"### {dataset_name} {estimator.upper()} - fold-only variation")
        report_parts.append(format_report_table(fold_table, "n_folds"))
    report_parts.append(f"### {dataset_name} KNN vs Lasso")
    report_parts.append(markdown_table(head_comparisons[dataset_name]))

report_path = repo_root / "results" / "scores" / "lfc_embedding_performance_report.md"
report_path.write_text("\n\n".join(report_parts) + "\n")
report_path


PosixPath('/Users/matthewmella/repos/foundation-models-perturbation/results/scores/lfc_embedding_performance_report.md')